# Enrichment analysis

In [35]:

import GSEA_suit as gsea
import pandas as pd
import numpy as np
import os
import getting_threshold_genes as gtg


In [36]:
def get_selected_genes(csv_file):
    rank_genes = gtg.get_gene_rank(csv_file)
    alpha = gtg.get_inflexion_point_from_gene_rank(rank_genes)
    gene_rank_df = rank_genes[rank_genes["abs_coef"]>alpha]
    return gene_rank_df

In [37]:

def get_selected_genes_format_RF(csv_file):
    rank_genes = gtg.get_gene_rank_format_RF(csv_file)
    alpha = gtg.get_inflexion_point_from_gene_rank(rank_genes)
    gene_rank_df = rank_genes[rank_genes["abs_coef"]>alpha]
    return gene_rank_df

In [48]:
def get_selected_genes_format_catBoost(csv_file):
    rank_genes = gtg.get_gene_rank_format_cat_temporal(csv_file)
    alpha = 0
    gene_rank_df = rank_genes[rank_genes["abs_coef"]>alpha]
    return gene_rank_df

In [81]:
algorithm = "ridge_L2_0"
#algorithm = "smote_catBoost"
gmt_database = "/home/karen/Documents/phd/Resources/Enrichment_dbs/HALLMARK_MYOGENESIS.v2023.2.Hs.gmt"

database = "Tabula_Sapiens"
database = gmt_database
#database = "KEGG_2021_Human"
#database = "SysMyo_Muscle_Gene_Sets"
#database = "WikiPathway_2023_Human"
#database = "GO_Molecular_Function_2023"

In [73]:
csv_path = f"Results/feature_selection/RNAseq/{algorithm}/"
file_list = os.listdir(csv_path)
file_list = [f for f in file_list if f.endswith('_Symbols.csv')]
file_list

['Experiment_GSE129643_feature_selection_Symbols.csv',
 'Experiment_GSE167186_feature_selection_Symbols.csv',
 'Experiment_GSE152558_feature_selection_Symbols.csv',
 'RNAseq_All_abundances_adjusted_feature_selection_Symbols.csv',
 'Sex_male_feature_selection_Symbols.csv',
 'Status_Sarcopenia_feature_selection_Symbols.csv',
 'Status_Healthy_feature_selection_Symbols.csv',
 'Experiment_GSE60590_feature_selection_Symbols.csv',
 'Experiment_GSE157585_feature_selection_Symbols.csv',
 'Experiment_GSE164471_feature_selection_Symbols.csv',
 'Sex_female_feature_selection_Symbols.csv',
 'Status_trained_feature_selection_Symbols.csv']

In [40]:
csv_file = 'RNAseq_All_abundances_adjusted_feature_selection_Symbols.csv'
csv_file

'RNAseq_All_abundances_adjusted_feature_selection_Symbols.csv'

In [88]:
sections = []

for csv_file in file_list:#[csv_file]:
    try:
        if "random_forest" in algorithm :
            rank_list = get_selected_genes_format_RF(csv_path + csv_file)
        elif "smote_c" in algorithm:
            rank_list = get_selected_genes_format_catBoost(csv_path + csv_file)
        else:
            rank_list = get_selected_genes(csv_path + csv_file)
        symbols = rank_list["Symbol"].tolist()
        #symbols = rank_list["symbol"].tolist()
        symbols = list(set(symbols))
        symbols=symbols[1:]
        coef = rank_list["coef"].tolist()
        #coef = rank_list["V1"].tolist()
        coef = list(set(coef))
        coef=coef[1:]
        dict_list = {"name": csv_file, "symbols": symbols, "coef": coef, "description": f"RNAseq {algorithm}"}
        
        sets_ranks = {}
        for gene, coef in zip(rank_list["Symbol"], rank_list["abs_coef"]):
            if gene in sets_ranks:
                sets_ranks[gene] += coef
            else:
                sets_ranks[gene] = coef
        sets_ranks = {k: v for k, v in sorted(sets_ranks.items(), key=lambda item: item[1], reverse=True)}
        # asing the value to the dictionary on the position they have
        for i, gene in enumerate(sets_ranks):
            sets_ranks[gene] = i+1
        folder_save = f"Results/feature_selection/RNAseq/{algorithm}/csv_file/"
        results = gsea.get_GSEA(rank_dict=sets_ranks, database= database, min_size=1, outdir=folder_save)
        enrichment = results.res2d
        h_enrichment = enrichment[enrichment['FDR q-val']<0.2]
        dict_list[database] = enrichment["Term"]
        sections.append(dict_list)

    except Exception as e:
        print("Error with file: ", csv_file, "error ", e)

# save sections
sections_df = pd.DataFrame(sections)
#sections_df.to_csv(f"Results/feature_selection/RNAseq/{algorithm}/{database}_selected_genes_all.csv", index=False)
sections_df

/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent w

Error with file:  ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 Status_Healthy_feature_selection_Symbols.csv error  SVD did not converge in Linear Least Squares
Error with file:  Experiment_GSE60590_feature_selection_Symbols.csv error  SVD did not converge in Linear Least Squares
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal val

/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent w

,name,symbols,coef,description,/home/karen/Documents/phd/Resources/Enrichment_dbs/HALLMARK_MYOGENESIS.v2023.2.Hs.gmt
0,Experiment_GSE129643_feature_selection_Symbols...,"[MTERF3, TRAJ39, ABCE1, YWHAH, STYX, ACSL1, DN...","[0.06874883466850641, -0.04945602400059417, 0....",RNAseq ridge_L2_0,0 HALLMARK_INFLAMMATORY_RESPONSE 1 ...
1,Experiment_GSE167186_feature_selection_Symbols...,"[MTERF3, ALG9, BTBD3, DNAJC2, ABCE1, TAB2, CAM...","[-0.05401227064808972, 0.04958098566201505, 0....",RNAseq ridge_L2_0,0 HALLMARK_INFLAMMATORY_RESPONSE 1 ...
2,Experiment_GSE152558_feature_selection_Symbols...,"[ZNF418, CAPNS1, MYH3, EIF3EP3, RANBP9, TRBV26...","[-0.006172467081098095, 0.01105299609572099, 0...",RNAseq ridge_L2_0,0 HALLMARK_DNA_REPAIR 1 ...
3,RNAseq_All_abundances_adjusted_feature_selecti...,"[MTERF3, SATL1, PATJ, PTMS, CAPZA2, HLA-B, IKB...","[0.1882640921440763, -0.18212858894399941, -1....",RNAseq ridge_L2_0,0 HALLMARK_INFLAMMATORY_RESPONSE 1 ...
4,Sex_male_feature_selection_Symbols.csv,"[SATL1, RNA5SP513, RNA5SP509, RNA5SP48, RNA5SP...","[-0.08499529972192735, 0.14201529597074714, 0....",RNAseq ridge_L2_0,0 HALLMARK_MYOGENESIS 1 ...
5,Status_Sarcopenia_feature_selection_Symbols.csv,"[TCERG1, ABHD1, SATL1, HSPB11, DNAJC2, TAB2, M...","[0.006759327195045021, 0.014183280598438692, 0...",RNAseq ridge_L2_0,0 HALLMARK_DNA_REPAIR 1 ...
6,Experiment_GSE157585_feature_selection_Symbols...,"[EGFL7, TXLNGY, HIGD1A, CCR10, RABEP2, PRRG1, ...","[-0.06730714876104968, 0.05630011864377938, 0....",RNAseq ridge_L2_0,0 HALLMARK_APOPTOSIS 1 HALLM...
7,Experiment_GSE164471_feature_selection_Symbols...,"[SATL1, MYH3, ANKRD23, ABCE1, MAST4, RNA5SP20,...","[0.24704592192589134, 0.1649144498630691, 0.10...",RNAseq ridge_L2_0,0 HALLMARK_DNA_REPAIR 1 ...
8,Sex_female_feature_selection_Symbols.csv,"[EIF3EP3, RNA5SP509, ABCE1, TBCE, TRAJ39, YWHA...","[-0.15791264946863784, 0.12392276183217599, 0....",RNAseq ridge_L2_0,0 HALLMARK_INFLAMMATORY_RESPONSE 1 ...
9,Status_trained_feature_selection_Symbols.csv,"[WNT11, NSL1, TTC23, COLQ, MTERF3, HSPB11, AKR...","[0.00045621832607647907, -0.000460894897275739...",RNAseq ridge_L2_0,0 HALLMARK_DNA_REPAIR 1 HALLM...


In [74]:
algorithm
rank_list
csv_path
csv_file = "RNAseq_All_abundances_adjusted_feature_selection_Symbols.csv"
rank_list = get_selected_genes(csv_path + csv_file)

/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":


In [76]:

sets_ranks = {}
for gene, coef in zip(rank_list["Symbol"], rank_list["coef"]):
    if gene in sets_ranks:
        sets_ranks[gene] += coef
    else:
        sets_ranks[gene] = coef
sets_ranks = {k: v for k, v in sorted(sets_ranks.items(), key=lambda item: item[1], reverse=True)}
sets_ranks

{'APOD': '0.6267889319434388',
 'METTL25': '0.6183406478418709',
 'RNA5SP482': '0.5964238444266273',
 'IFITM1': '0.5915365362312107',
 'HCN1': '0.5618525227180531',
 'MAB21L3': '0.5168269324181688',
 'MTRNR2L8': '0.4577738694115041',
 'TRAJ21': '0.43970394873164187',
 'ST7': '0.43310391117567526',
 'PIK3R1': '0.41494000710154033',
 'UQCRH': '0.40416755105427715',
 'PPDPF': '0.39705713298830414',
 'NCOA2': '0.3900218681656765',
 'ABLIM2': '0.38260048553828063',
 'C3': '0.3747153462732006',
 'KCNMA1': '0.37088737652489245',
 'PKD1P4': '0.35789409501255254',
 'PSMG4': '0.3536770588806263',
 'RNA5SP497': '0.3515611232530883',
 'RERE': '0.33709801293935987',
 'SMBD1P': '0.337058088001567',
 'RNA5SP249': '0.33318264878959736',
 'RNA5SP287': '0.3320683194874301',
 'TSPAN7': '0.33199499256260195',
 'UCP2': '0.32076750668830745',
 'BACE1': '0.3207045120317765',
 'CYP2C9': '0.31840065013926466',
 'EIF5A': '0.3173591520518855',
 'DCUN1D1': '0.3173002182850625',
 'TRAJ9': '0.3172419878643793',
 'G

In [ ]:
# asing the value to the dictionary on the position they have
for i, gene in enumerate(sets_ranks):
    sets_ranks[gene] = i+1

In [78]:
folder_save = f"Results/feature_selection/RNAseq/{algorithm}/csv_file/"

enrichment = gsea.get_GSEA(rank_dict=sets_ranks, database= gmt_database, min_size=5, outdir=folder_save).res2d

/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.10/site-packages/gseapy/gsea.py:427: RuntimeWarning: invalid value encountered in scalar divide
  dups = rank_metric.apply(lambda df: df.duplicated().sum() / df.size)


In [80]:
algorithm

'ridge_L2_0'

In [79]:
enrichment

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes


In [ ]:
h_enrichment#["Term"]

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes


In [85]:
list(enrichment['Term'])

['HALLMARK_MYOGENESIS']

In [ ]:
sets_ranks

{0: 1,
 'KDM5D': 2,
 'EIF1AY': 3,
 'RPS4Y1': 4,
 'GGT7': 5,
 'NPY6R': 6,
 'EGLN3': 7,
 'PUDP': 8,
 'TTTY14': 9,
 'USP9Y': 10,
 'PRKAG3': 11,
 'CAVIN4': 12,
 'TXLNGY': 13,
 'C11orf71': 14,
 'ADAMTSL4': 15,
 'CNNM4': 16,
 'PANK1': 17,
 'CNNM3': 18,
 'PFKFB2': 19,
 'ALDH6A1': 20,
 'ZNF358': 21,
 'MIDN': 22,
 'PATJ': 23,
 'TMEM94': 24,
 'DAPK3': 25,
 'PIK3R1': 26,
 'SNRPN': 27,
 'SLC25A26': 28,
 'NDRG4': 29,
 'CIBAR1': 30,
 'ACTN3': 31,
 'PCGF3': 32,
 'RPL26P6': 33,
 'HPN': 34,
 'UTY': 35,
 'SRPK3': 36,
 'SPATS2L': 37,
 'MOB3C': 38,
 'ILRUN': 39,
 'C4orf54': 40,
 'SLC25A25': 41,
 'GSTM3': 42,
 'RYR3': 43,
 'NMNAT1': 44,
 'ST7': 45,
 'METTL7A': 46,
 'FAH': 47,
 'PTDSS1': 48,
 'C12orf75': 49,
 'PPP1R3B': 50,
 'SLC37A4': 51,
 'CHRDL2': 52,
 'CDK18': 53,
 'ABHD18': 54,
 'DROSHA': 55,
 'LMO1': 56,
 'EIF6': 57,
 'LONRF1': 58,
 'PLAAT3': 59,
 'SLC27A1': 60,
 'CBY1': 61,
 'NEIL2': 62,
 'HYAL1': 63,
 'FRMD6': 64,
 'SRSF1': 65,
 'KIAA1671': 66,
 'VPS35L': 67,
 'FAM118A': 68,
 'CCDC57': 69,
 'SHISA4'

In [ ]:
lab_mirna_targets= ["gap43",
"acvr2a",
"hdac9",
"grp78",
"atf6",
"pgc1-a",
"acvr1b",
"NCAM",
"Hmbox1",
"p2ry6",
"camk2a",
"cadps",
"lch1",
"Il1rapl1",
"Carf",
"Aak1",
"P2ry1",
"Yy1",
"Psme3",
"Bach2",
"Smurf2",
"Atp6v1g1",
"Kcnk10",
"Gpx4",
"Psmg4",
"bach2",
"pten",
"p62",
"Prdx6"]

In [87]:
sections_df

,name,symbols,coef,description,/home/karen/Documents/phd/Resources/Enrichment_dbs/HALLMARK_MYOGENESIS.v2023.2.Hs.gmt
0,Experiment_GSE129643_feature_selection_Symbols...,"[MTERF3, TRAJ39, ABCE1, YWHAH, STYX, ACSL1, DN...","[0.06874883466850641, -0.04945602400059417, 0....",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."
1,Experiment_GSE167186_feature_selection_Symbols...,"[MTERF3, ALG9, BTBD3, DNAJC2, ABCE1, TAB2, CAM...","[-0.05401227064808972, 0.04958098566201505, 0....",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."
2,Experiment_GSE152558_feature_selection_Symbols...,"[ZNF418, CAPNS1, MYH3, EIF3EP3, RANBP9, TRBV26...","[-0.006172467081098095, 0.01105299609572099, 0...",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."
3,RNAseq_All_abundances_adjusted_feature_selecti...,"[MTERF3, SATL1, PATJ, PTMS, CAPZA2, HLA-B, IKB...","[0.1882640921440763, -0.18212858894399941, -1....",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."
4,Sex_male_feature_selection_Symbols.csv,"[SATL1, RNA5SP513, RNA5SP509, RNA5SP48, RNA5SP...","[-0.08499529972192735, 0.14201529597074714, 0....",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."
5,Status_Sarcopenia_feature_selection_Symbols.csv,"[TCERG1, ABHD1, SATL1, HSPB11, DNAJC2, TAB2, M...","[0.006759327195045021, 0.014183280598438692, 0...",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."
6,Experiment_GSE157585_feature_selection_Symbols...,"[EGFL7, TXLNGY, HIGD1A, CCR10, RABEP2, PRRG1, ...","[-0.06730714876104968, 0.05630011864377938, 0....",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."
7,Experiment_GSE164471_feature_selection_Symbols...,"[SATL1, MYH3, ANKRD23, ABCE1, MAST4, RNA5SP20,...","[0.24704592192589134, 0.1649144498630691, 0.10...",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."
8,Sex_female_feature_selection_Symbols.csv,"[EIF3EP3, RNA5SP509, ABCE1, TBCE, TRAJ39, YWHA...","[-0.15791264946863784, 0.12392276183217599, 0....",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."
9,Status_trained_feature_selection_Symbols.csv,"[WNT11, NSL1, TTC23, COLQ, MTERF3, HSPB11, AKR...","[0.00045621832607647907, -0.000460894897275739...",RNAseq ridge_L2_0,"0 HALLMARK_MYOGENESIS Name: Term, dtype: ob..."


In [59]:
sections_df["Tabula_Sapiens"][3]

0                                 Trachea-ciliated Cell
1                             Eye-retinal Ganglion Cell
2                               Thymus-fast Muscle Cell
3                               Spleen-endothelial Cell
4                       Uterus-ciliated Epithelial Cell
5                                      Muscle-mast Cell
6                                         Liver-nk Cell
7     Small Intestine-transit Amplifying Cell Of Sma...
8                   Eye-epithelial Cell Of Lacrimal Sac
9                         Pancreas-pancreatic Beta Cell
10                              Eye-limbal Stromal Cell
11                          Pancreas-pancreatic Pp Cell
12                                    Muscle-macrophage
13                                       Fat-macrophage
14                          Large Intestine-goblet Cell
15         Prostate-luminal Cell Of Prostate Epithelium
16                                     Eye-ciliary Body
17                  Bone Marrow-hematopoietic St

In [51]:
for row in sections_df.iterrows():
    symbols = row[1]["symbols"]
    known_genes = [gene for gene in symbols if gene in lab_mirna_targets]
    print (row[1]["name"], len(known_genes))

Experiment_GSE167186_feature_selection_Symbols.csv 0
Sex_male_feature_selection_Symbols.csv 0
Status_Sarcopenia_feature_selection_Symbols.csv 0
Status_Healthy_feature_selection_Symbols.csv 0
feature_importance_Symbols.csv 0
Experiment_GSE60590_feature_selection_Symbols.csv 0
Experiment_GSE164471_feature_selection_Symbols.csv 0
Sex_female_feature_selection_Symbols.csv 0


In [ ]:
symbols

['WNT11',
 'NSL1',
 'TTC23',
 'COLQ',
 'MTERF3',
 'HSPB11',
 'AKR1B15',
 'LACTB2',
 'CBX7',
 'PATJ',
 'SLC25A34',
 'ACSL1',
 'TXLNGY',
 'RSAD1',
 'C11orf71',
 'DROSHA',
 'EFHD1',
 'HLA-B',
 'IKBKB',
 'MIDN',
 'SAFB',
 'TLCD3B',
 'PI4KB',
 'PRKCH',
 'VPS26B',
 'ND4L',
 'SYNJ2BP-COX16',
 'NUDT16L1',
 'SNAP29',
 'ALDH2',
 'RNA5SP37',
 'GGPS1',
 'ALKBH7',
 'FAM238C',
 'ZNF362',
 'LMBR1',
 'PARD3',
 'LRRC23',
 'U2AF1',
 'CCDC141',
 'PARVB',
 'TBRG4',
 'CYP3A54P',
 'STYXL2',
 'DDHD2',
 'ZNF598',
 'TEX30',
 'UBE2D4',
 'SLC37A4',
 'YWHAB',
 'KCNT1',
 'LRMDA',
 'NFKBIA',
 'BAX',
 'PFKFB1',
 'DECR1',
 'ATP8',
 'GGA1',
 'YIPF7',
 'PLEC',
 'FRMD6',
 'DDX24',
 'FBXO3',
 'USF2',
 'NAT8L',
 'ADCY3',
 'MRPL27',
 'NUDT2',
 'E2F8',
 'C4orf54',
 'TRBJ2-6',
 'TMEM38A',
 'MRPL48',
 'ADAMTSL4',
 'RTN3',
 'UBE2V2',
 'DYRK1B',
 'FLYWCH1',
 'HES4',
 'CSRNP2',
 'MPP7',
 'SPAG9',
 'SSH1',
 'EIF3C',
 'NR1H3',
 'POLR3H',
 'TNIP2',
 'TIMM10',
 'AMPD3',
 'KLHL41',
 'YJU2B',
 'AP4M1',
 'YAF2',
 'LMO7',
 'USP24',
 'HN

In [57]:
repeated_terms = []
for row in sections_df.iterrows():
    terms = row[1][database]
    repeated_terms.append(set(terms))
repeated_terms

[{'Blood-erythrocyte',
  'Bone Marrow-erythrocyte',
  'Bone Marrow-nampt Neutrophil',
  'Bone Marrow-neutrophil',
  'Eye-erythroid Lineage Cell',
  'Eye-macrophage',
  'Fat-macrophage',
  'Large Intestine-mature Enterocyte',
  'Large Intestine-monocyte',
  'Lung-capillary Aerocyte',
  'Lung-neutrophil',
  'Lymph Node-macrophage',
  'Prostate-basal Cell Of Prostate Epithelium',
  'Prostate-erythroid Progenitor',
  'Salivary Gland-monocyte',
  'Small Intestine-monocyte',
  'Thymus-erythrocyte',
  'Tongue-pericyte Cell',
  'Trachea-neutrophil',
  'Vasculature-macrophage'},
 {'Fat-smooth Muscle Cell',
  'Salivary Gland-adventitial Cell',
  'Salivary Gland-monocyte',
  'Salivary Gland-pericyte Cell',
  'Thymus-vascular Associated Smooth Muscle Cell',
  'Vasculature-smooth Muscle Cell'},
 {'Thymus-erythrocyte'},
 {'Blood-basophil',
  'Blood-hematopoietic Stem Cell',
  'Bone Marrow-hematopoietic Stem Cell',
  'Bone Marrow-myeloid Progenitor',
  'Eye-ciliary Body',
  'Eye-epithelial Cell Of La

In [58]:
# count repetiton of terms
from collections import Counter
count_terms = Counter()
for terms in repeated_terms:
    count_terms.update(terms)
count_terms

Counter({'Thymus-erythrocyte': 4,
         'Prostate-basal Cell Of Prostate Epithelium': 3,
         'Tongue-pericyte Cell': 3,
         'Large Intestine-mature Enterocyte': 3,
         'Blood-erythrocyte': 3,
         'Bone Marrow-erythrocyte': 3,
         'Fat-smooth Muscle Cell': 3,
         'Vasculature-smooth Muscle Cell': 3,
         'Salivary Gland-pericyte Cell': 3,
         'Prostate-endothelial Cell': 3,
         'Thymus-fast Muscle Cell': 3,
         'Fat-macrophage': 2,
         'Small Intestine-monocyte': 2,
         'Bone Marrow-nampt Neutrophil': 2,
         'Eye-erythroid Lineage Cell': 2,
         'Lung-capillary Aerocyte': 2,
         'Salivary Gland-monocyte': 2,
         'Large Intestine-monocyte': 2,
         'Prostate-erythroid Progenitor': 2,
         'Thymus-vascular Associated Smooth Muscle Cell': 2,
         'Salivary Gland-adventitial Cell': 2,
         'Eye-ciliary Body': 2,
         'Liver-nk Cell': 2,
         'Small Intestine-transit Amplifying Cell Of Sm

In [53]:
len(repeated_terms)

8

In [54]:
intersection = set.intersection(*repeated_terms)
intersection

set()

In [56]:
list(intersection)

[]

In [34]:
list(intersection)

['Salivary Gland-myoepithelial Cell',
 'Fat-macrophage',
 'Muscle-smooth Muscle Cell',
 'Lymph Node-stromal Cell',
 'Eye-ciliary Body',
 'Fat-smooth Muscle Cell',
 'Lung-basal Cell',
 'Large Intestine-mature Enterocyte',
 'Thymus-medullary Thymic Epithelial Cell',
 'Prostate-neutrophil',
 'Salivary Gland-adventitial Cell',
 'Trachea-fibroblast',
 'Blood-erythrocyte',
 'Lung-type I Pneumocyte',
 'Muscle-erythrocyte',
 'Vasculature-lymphatic Endothelial Cell',
 'Eye-macrophage',
 'Muscle-capillary Endothelial Cell',
 'Muscle-fast Muscle Cell',
 'Salivary Gland-ionocyte',
 'Prostate-smooth Muscle Cell',
 'Vasculature-smooth Muscle Cell',
 'Kidney-kidney Epithelial Cell',
 'Large Intestine-immature Enterocyte',
 'Muscle-endothelial Cell Of Lymphatic Vessel',
 'Muscle-endothelial Cell Of Artery',
 'Uterus-vascular Associated Smooth Muscle Cell',
 'Small Intestine-monocyte',
 'Fat-neutrophil',
 'Skin-cell Of Skeletal Muscle',
 'Bladder-fibroblast',
 'Thymus-fast Muscle Cell',
 'Fat-fibroblas

In [55]:
len(intersection)

0